In [19]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_module import PhoDataset, Vocabulary
from src.models.model02 import Encoder, Decoder, Attention, Model02

In [20]:
from typing import Dict
from torch.nn.utils.rnn import pad_sequence

In [21]:
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
import pandas as pd
from tqdm import tqdm

from torchmetrics.text.rouge import ROUGEScore

# 1. Preparation

## 1.1. Vocabulary

In [22]:
vocab = Vocabulary(
    r'..\data\small-train.json',
    freq_threshold=1
)

                                                 english  \
0                           It begins with a countdown .   
1      On August 14th , 1947 , a woman in Bombay goes...   
2      Across India , people hold their breath for th...   
3      And at the stroke of midnight , a squirming in...   
4      These events form the foundation of " Midnight...   
...                                                  ...   
19995               And the man was incredibly curious .   
19996  And he wanted to understand what it was and wh...   
19997                    And one day , we were walking .   
19998               We were in France , in Les Houches .   
19999               We were up in the mountains , 1976 .   

                                              vietnamese  
0             Câu chuyện bắt đầu với buổi lễ đếm ngược .  
1      Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở...  
2      Cùng lúc , trên khắp đất Ấn , người ta nín thở...  
3      Khi đồng hồ điểm thời khắc nửa đêm ,

## 1.2. Model

In [23]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


### 1.2.1. `Encoder`, `Attention` and `Decoder`

In [24]:
INPUT_DIM = vocab.num_en_words
OUTPUT_DIM = vocab.num_vi_words

ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 256
N_LAYERS = 3
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [25]:
encoder = Encoder(
    input_dim=INPUT_DIM, 
    emb_dim=ENC_EMB_DIM, 
    hid_dim=HID_DIM, 
    n_layers=N_LAYERS, 
    dropout=ENC_DROPOUT
)
print(encoder)

Encoder(
  (embedding): Embedding(17418, 256)
  (rnn): LSTM(256, 256, num_layers=3, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=512, out_features=256, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [26]:
attention = Attention(HID_DIM)
print(attention)

Attention(
  (attn): Linear(in_features=768, out_features=256, bias=True)
  (v): Linear(in_features=256, out_features=1, bias=False)
)


In [27]:
decoder = Decoder(
    output_dim=OUTPUT_DIM, 
    emb_dim=DEC_EMB_DIM, 
    hid_dim=HID_DIM, 
    n_layers=N_LAYERS, 
    dropout=DEC_DROPOUT, 
    attention=attention
)
print(decoder)

Decoder(
  (attention): Attention(
    (attn): Linear(in_features=768, out_features=256, bias=True)
    (v): Linear(in_features=256, out_features=1, bias=False)
  )
  (embedding): Embedding(7061, 256)
  (rnn): LSTM(768, 256, num_layers=3, dropout=0.5)
  (fc_out): Linear(in_features=1024, out_features=7061, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


### 1.2.2. Model with `Encoder`, `Attention` and `Decoder`

In [28]:
model = Model02(encoder, decoder, device).to(device)
print(model)

Model02(
  (encoder): Encoder(
    (embedding): Embedding(17418, 256)
    (rnn): LSTM(256, 256, num_layers=3, dropout=0.5, bidirectional=True)
    (fc): Linear(in_features=512, out_features=256, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (attention): Attention(
      (attn): Linear(in_features=768, out_features=256, bias=True)
      (v): Linear(in_features=256, out_features=1, bias=False)
    )
    (embedding): Embedding(7061, 256)
    (rnn): LSTM(768, 256, num_layers=3, dropout=0.5)
    (fc_out): Linear(in_features=1024, out_features=7061, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)


## 1.3. Dataset

In [29]:
def collate_fn(
    batch: Dict,
    padding_value: int=0
):
    batch_en = []
    batch_vi = []
    for item in batch:
        batch_en.append(item[0])
        batch_vi.append(item[1])

    en_padded = pad_sequence(batch_en, batch_first=False, padding_value=padding_value)
    vi_padded = pad_sequence(batch_vi, batch_first=False, padding_value=padding_value)

    en_lengths = torch.tensor([len(x) for x in batch_en])
    vi_lengths = torch.tensor([len(x) for x in batch_vi])
    
    return {
        'encoder_input': en_padded,
        'decoder_input': vi_padded,
        'en_lengths': en_lengths,
        'vi_lengths': vi_lengths
    }

In [30]:
def my_collate_fn(batch):
    return collate_fn(batch, padding_value=vocab.en2i['<PAD>'])

In [31]:
train_df = PhoDataset(
    '..\data\small-train.json',
    vocab
)
train_loader = DataLoader(
    train_df,
    batch_size = 64,
    shuffle=True,
    collate_fn=my_collate_fn,
    num_workers=0,
    drop_last=True,
)

In [32]:
dev_df = PhoDataset(
    '..\data\small-dev.json',
    vocab
)
dev_loader = DataLoader(
    dev_df,
    batch_size = 64,
    shuffle=False,
    collate_fn=my_collate_fn,
    num_workers=0,
    drop_last=True,
)

In [33]:
test_df = PhoDataset(
    '..\data\small-test.json',
    vocab
)
test_loader = DataLoader(
    test_df,
    batch_size = 64,
    shuffle=False,
    collate_fn=my_collate_fn,
    num_workers=0,
    drop_last=True,
)

# 2. Training

## 2.1. Epoch

### 2.1.1. Epoch training

In [34]:
criterion = nn.CrossEntropyLoss(ignore_index=vocab.vi2i['<PAD>'])
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [35]:
def train_epoch(model, loader, optimizer, criterion, device, clip=1.0):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(loader, desc='Training', leave=True)
    
    for batch in progress_bar:
        src = batch['encoder_input'].to(device)
        trg = batch['decoder_input'].to(device)
        
        optimizer.zero_grad()
        trg_input = trg[:-1, :] 
        
        trg_target = trg[1:, :] 
        output = model(src, trg_input)
        
        output_dim = output.shape[-1]
        output = output.reshape(-1, output_dim)
        loss = criterion(output, trg_target.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
        
    return epoch_loss / len(loader)

### 2.1.2. Epoch evaluating

In [36]:
def evaluate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        progress_bar = tqdm(loader, desc='Evaluating', leave=True)
        
        for batch in progress_bar:
            src = batch['encoder_input'].to(device)
            trg = batch['decoder_input'].to(device)

            trg_input = trg[:-1, :]
            trg_target = trg[1:, :]

            output = model(src, trg_input, teacher_forcing_ratio=0)
            
            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim)
            
            loss = criterion(output, trg_target.reshape(-1))
            epoch_loss += loss.item()
            
            progress_bar.set_postfix({'val_loss': loss.item()})
            
    return epoch_loss / len(loader)

## 2.2. Training model

In [37]:
N_EPOCHS = 10

In [38]:
def training(
    model,
    train_loader,
    dev_loader,
    optimizer,
    criterion,
    device,
    num_epochs = 10,
    best_valid_loss = float('inf')
):
    training_loss = []
    evaluating_loss = []

    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        valid_loss = evaluate(model, dev_loader, criterion, device)
        
        training_loss.append(train_loss)
        evaluating_loss.append(valid_loss)
        
        print(f'\n' + '='*30)
        print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.4f} | Val. Loss: {valid_loss:.4f}')
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            import os
            os.makedirs('../checkpoints/', exist_ok=True)
            
            torch.save(model.state_dict(), '../checkpoints/best_model02.pt')
            print(f"\t [!] New Best Valid Loss. Model saved.")
        print('='*30 + '\n')

        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return training_loss, evaluating_loss

In [ ]:
result = training(model, train_loader, dev_loader, optimizer, criterion, device)

## 2.3. Finetuning trained model

In [40]:
def finetune(
    model, 
    train_loader, 
    dev_loader, 
    criterion, 
    device, 
    num_epochs=5,
    learning_rate=0.002,
    save_path='../checkpoints/finetuned_model.pt'
):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_valid_loss = float('inf')
    ft_train_losses = []
    ft_val_losses = []
    
    print(f"Starting Fine-tuning with LR={learning_rate}...")
    
    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        valid_loss = evaluate(model, dev_loader, criterion, device)
        ft_train_losses.append(train_loss)
        ft_val_losses.append(valid_loss)
        
        print(f'\nEpoch: {epoch+1:02} | FT Train Loss: {train_loss:.4f} | FT Val. Loss: {valid_loss:.4f}')

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), save_path)
            print(f"\t [!] New Best Fine-tuned Model saved.")
        print('='*30)
        
    return ft_train_losses, ft_val_losses

In [41]:
model = Model02(encoder, decoder, device).to(device)
model.load_state_dict(torch.load('../checkpoints/best_model02.pt'))

criterion = nn.CrossEntropyLoss(ignore_index=vocab.vi2i['<PAD>'])

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_26000\487228330.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('../checkpoints/best_model0

In [ ]:
history = finetune(
    model, 
    train_loader, 
    dev_loader, 
    criterion, 
    device,
    num_epochs=5,
    learning_rate=0.002,
    save_path='../checkpoints/finetuned_model02.pt'
)

Starting Fine-tuning with LR=0.002...


Training:   8%|▊         | 25/312 [01:00<10:20,  2.16s/it, loss=5.89]

# 3. Evaluation

In [ ]:
def evaluate_rouge(model, loader, vocab, device, max_len=50):
    model.eval()
    rouge_metric = ROUGEScore(rouge_keys='rougeL')
    
    predictions = []
    references = []
    progress_bar = tqdm(loader, desc='Calculating ROUGE-L', leave=True)

    with torch.no_grad():
        for batch in progress_bar:
            src = batch['encoder_input'].to(device)
            trg = batch['decoder_input']
            
            for i in range(src.shape[0]):
                single_src = src[i:i+1]
                
                hidden, cell = model.encoder(single_src)
                
                trg_indexes = [vocab.vi2i['<SOS>']]
                
                for _ in range(max_len):
                    trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
                    output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
                    pred_token = output.argmax(1).item()
                    trg_indexes.append(pred_token)
                    if pred_token == vocab.vi2i['<EOS>']:
                        break
                
                def tokens_to_sentence(ids, vocabulary, is_vi=True):
                    specials = {vocabulary.vi2i['<SOS>'], vocabulary.vi2i['<EOS>'], vocabulary.vi2i['<PAD>']}
                    if is_vi:
                        return " ".join([vocabulary.i2vi[idx] for idx in ids if idx not in specials])
                    else:
                        return " ".join([vocabulary.i2en[idx] for idx in ids if idx not in specials])

                pred_sentence = tokens_to_sentence(trg_indexes, vocab)
                actual_sentence = tokens_to_sentence(trg[i].tolist(), vocab)
                
                predictions.append(pred_sentence)
                references.append(actual_sentence)

    results = rouge_metric(predictions, references)
    return results['rougeL_fmeasure'].item()